In [1]:
import kagglehub
import pandas as pd
import os
import re
import nltk
nltk.download('stopwords')
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation, NMF

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/konstantinhanemann/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [2]:
# Load Dataset
path = "/Users/konstantinhanemann/Desktop/Studium/2026 - 01 - Project Data Analysis/dataverse_files"

# Find csv-file
csv_files = [f for f in os.listdir(path) if f.endswith('.csv')]
df = pd.concat([
    pd.read_csv(os.path.join(path, f), encoding='latin1', low_memory=False) 
    for f in csv_files
], ignore_index=True)

In [3]:
# 1. Show first lines
print("--- FIRST ROWS ---")
print(df.head())

--- FIRST ROWS ---
  CALLERTYPE DATECANCELLED           DATEINVTDONE         DATETIMECLOSED  \
0        NaN           NaN  2008-09-09 00:00:00.0  2008-09-09 12:48:00.0   
1        NaN           NaN  2008-09-11 00:00:00.0  2008-09-11 11:30:12.0   
2        NSO           NaN  2008-09-09 00:00:00.0  2008-09-11 10:36:28.0   
3        NSO           NaN  2008-09-09 00:00:00.0  2008-09-09 11:07:20.0   
4        NaN           NaN  2008-09-08 00:00:00.0  2008-09-22 14:04:37.0   

            DATETIMEINIT           DESCRIPTION  \
0  2008-09-02 10:34:20.0  Sign needs attention   
1  2008-09-05 10:28:03.0          Light(s) Out   
2  2008-09-05 13:22:16.0  Sign needs attention   
3  2008-09-05 13:27:04.0        Signal Damaged   
4  2008-09-08 08:20:00.0      Signals Flashing   

                                         EXPLANATION      EXPR1  \
0  Select this option for non-critical signs (str...        NaN   
1                                                NaN  St. Louis   
2  Select this option 

In [4]:
# 2. Find structure and data types
print("\n--- DATA INFO ---")
print(df.info())


--- DATA INFO ---
<class 'pandas.DataFrame'>
RangeIndex: 924577 entries, 0 to 924576
Data columns (total 22 columns):
 #   Column                              Non-Null Count   Dtype  
---  ------                              --------------   -----  
 0   CALLERTYPE                          713725 non-null  str    
 1   DATECANCELLED                       24805 non-null   str    
 2   DATEINVTDONE                        883090 non-null  str    
 3   DATETIMECLOSED                      872286 non-null  str    
 4   DATETIMEINIT                        924577 non-null  str    
 5   DESCRIPTION                         924568 non-null  str    
 6   EXPLANATION                         244855 non-null  str    
 7   EXPR1                               129569 non-null  str    
 8   GROUP                               924576 non-null  str    
 9   NEIGHBORHOOD                        880825 non-null  object 
 10  PLAIN_ENGLISH_NAME_FOR_PROBLEMCODE  889074 non-null  str    
 11  PRJCOMPLETEDATE   

In [5]:
# 3. Identify column names
print("\n--- COLUMNS ---")
print(df.columns)


--- COLUMNS ---
Index(['CALLERTYPE', 'DATECANCELLED', 'DATEINVTDONE', 'DATETIMECLOSED',
       'DATETIMEINIT', 'DESCRIPTION', 'EXPLANATION', 'EXPR1', 'GROUP',
       'NEIGHBORHOOD', 'PLAIN_ENGLISH_NAME_FOR_PROBLEMCODE', 'PRJCOMPLETEDATE',
       'PROBADDRESS', 'PROBADDTYPE', 'PROBLEMCODE', 'PROBZIP', 'REQUESTID',
       'SRX', 'SRY', 'STATUS', 'SUBMITTO', 'WARD'],
      dtype='str')


In [6]:
# 4. Check for missing values
print("\n--- MISSING VALUES ---")
print(df.isnull().sum())


--- MISSING VALUES ---
CALLERTYPE                            210852
DATECANCELLED                         899772
DATEINVTDONE                           41487
DATETIMECLOSED                         52291
DATETIMEINIT                               0
DESCRIPTION                                9
EXPLANATION                           679722
EXPR1                                 795008
GROUP                                      1
NEIGHBORHOOD                           43752
PLAIN_ENGLISH_NAME_FOR_PROBLEMCODE     35503
PRJCOMPLETEDATE                        10002
PROBADDRESS                              977
PROBADDTYPE                                0
PROBLEMCODE                                0
PROBZIP                               602439
REQUESTID                                  0
SRX                                    17951
SRY                                    17945
STATUS                                   106
SUBMITTO                                 283
WARD                           

In [7]:
# 5. Remove entries of "description" with 0 values
df = df.dropna(subset=['DESCRIPTION'])

In [8]:
print(df['SRX'].value_counts().head(5))

SRX
904955.770    708
881862.750    335
881824.589    242
887471.250    168
881829.275    135
Name: count, dtype: int64


In [14]:
import pandas as pd
import os
import matplotlib.pyplot as plt

path = "/Users/konstantinhanemann/Downloads/archive/"

# path = "/Users/konstantinhanemann/Downloads/archive"
csv_path = os.path.join(path, "twitter_dataset.csv")

df = pd.read_csv(csv_path)

# Basic Inspection
print(df.info())
print(df.head())

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   Tweet_ID   10000 non-null  int64
 1   Username   10000 non-null  str  
 2   Text       10000 non-null  str  
 3   Retweets   10000 non-null  int64
 4   Likes      10000 non-null  int64
 5   Timestamp  10000 non-null  str  
dtypes: int64(3), str(3)
memory usage: 468.9 KB
None
   Tweet_ID        Username  \
0         1         julie81   
1         2   richardhester   
2         3  williamsjoseph   
3         4     danielsmary   
4         5      carlwarren   

                                                Text  Retweets  Likes  \
0  Party least receive say or single. Prevent pre...         2     25   
1  Hotel still Congress may member staff. Media d...        35     29   
2  Nice be her debate industry that year. Film wh...        51     25   
3  Laugh explain situation career occur serious. ...        37     1